In [25]:
import pytorch_kinematics as pk 
import numpy as np 
import torch
import time

In [26]:
hand_urdf_path = "/home/haoran_zheng/Documents/LfD_tactile/Avatar_Dex/avatar_dex/robot/assets/allegro_hand_right_no_collision.urdf"
chain = pk.build_chain_from_urdf(open(hand_urdf_path).read()).to(dtype=torch.float, device="cuda:0")
# chain = pk.build_serial_chain_from_urdf(open(hand_urdf_path).read(), end_link_name="link_3.0_tip", root_link_name="base_link").to(dtype=torch.float, device="cuda:0")
hand_mjcf_path = "/home/haoran_zheng/Documents/LfD_tactile/Avatar_Dex/avatar_dex/components/simulation/xml/right_hand.xml"
# chain = pk.build_chain_from_mjcf(open(hand_mjcf_path).read()).to(dtype=torch.float, device="cuda:0")

In [27]:
type(chain)

pytorch_kinematics.chain.Chain

In [28]:
chain.get_frame_names(exclude_fixed=False)

['base_link_frame',
 'link_0.0_frame',
 'link_1.0_frame',
 'link_2.0_frame',
 'link_3.0_frame',
 'link_3.0_tip_frame',
 'link_4.0_frame',
 'link_5.0_frame',
 'link_6.0_frame',
 'link_7.0_frame',
 'link_7.0_tip_frame',
 'link_8.0_frame',
 'link_9.0_frame',
 'link_10.0_frame',
 'link_11.0_frame',
 'link_11.0_tip_frame',
 'link_12.0_frame',
 'link_13.0_frame',
 'link_14.0_frame',
 'link_15.0_frame',
 'link_15.0_tip_frame',
 'palm_frame']

In [29]:
chain.get_frame_names()

['link_0.0_frame',
 'link_1.0_frame',
 'link_2.0_frame',
 'link_3.0_frame',
 'link_4.0_frame',
 'link_5.0_frame',
 'link_6.0_frame',
 'link_7.0_frame',
 'link_8.0_frame',
 'link_9.0_frame',
 'link_10.0_frame',
 'link_11.0_frame',
 'link_12.0_frame',
 'link_13.0_frame',
 'link_14.0_frame',
 'link_15.0_frame']

In [30]:
chain.get_joint_parameter_names()

['joint_0.0',
 'joint_1.0',
 'joint_2.0',
 'joint_3.0',
 'joint_4.0',
 'joint_5.0',
 'joint_6.0',
 'joint_7.0',
 'joint_8.0',
 'joint_9.0',
 'joint_10.0',
 'joint_11.0',
 'joint_12.0',
 'joint_13.0',
 'joint_14.0',
 'joint_15.0']

In [31]:
ALLEGRO_HOME_POSITION = [ 0., -0.17453293, 0.78539816, 0.78539816, 0., -0.17453293, 0.78539816, 0.78539816, 0.08726646, -0.08726646, 0.87266463, 0.78539816, 1.04719755, 0.43633231, 0.26179939, 0.78539816]

In [32]:
qpos = torch.tensor(ALLEGRO_HOME_POSITION, dtype=torch.float, device="cuda:0")
ts = time.time()
ret = chain.forward_kinematics(qpos)
# m = tg.get_ma
print("time: ", time.time() - ts)

time:  0.008094549179077148


In [33]:
ret[]

{'base_link': Transform3d(rot=tensor([[1., 0., 0., 0.]], device='cuda:0'), pos=tensor([[0., 0., 0.]], device='cuda:0')),
 'link_0.0': Transform3d(rot=tensor([[ 0.9990, -0.0436,  0.0000,  0.0000]], device='cuda:0'), pos=tensor([[ 0.0000,  0.0435, -0.0015]], device='cuda:0')),
 'link_1.0': Transform3d(rot=tensor([[ 0.9952, -0.0435, -0.0871,  0.0038]], device='cuda:0'), pos=tensor([[0.0000, 0.0449, 0.0148]], device='cuda:0')),
 'link_2.0': Transform3d(rot=tensor([[ 0.9528, -0.0416,  0.3004, -0.0131]], device='cuda:0'), pos=tensor([[-0.0094,  0.0496,  0.0678]], device='cuda:0')),
 'link_3.0': Transform3d(rot=tensor([[ 0.7653, -0.0334,  0.6422, -0.0280]], device='cuda:0'), pos=tensor([[0.0126, 0.0523, 0.0991]], device='cuda:0')),
 'link_3.0_tip': Transform3d(rot=tensor([[ 0.7653, -0.0334,  0.6422, -0.0280]], device='cuda:0'), pos=tensor([[0.0488, 0.0529, 0.1055]], device='cuda:0')),
 'link_4.0': Transform3d(rot=tensor([[1., 0., 0., 0.]], device='cuda:0'), pos=tensor([[0.0000, 0.0000, 0.0007

In [55]:
# get position and rotation
mat = ret["link_15.0_tip"].get_matrix()
rot = mat[:, :3, :3]
print(rot)
mat.inverse()[:, :3, :3]
trans = mat[:, :3, 3]
print(trans.shape)

tensor([[[-0.6443,  0.4532,  0.6160],
         [-0.5742, -0.8187,  0.0017],
         [ 0.5051, -0.3526,  0.7877]]], device='cuda:0')
torch.Size([1, 3])


In [35]:
import sapien.core as sapien

hand_urdf_path = "/home/haoran_zheng/Documents/LfD_tactile/Avatar_Dex/avatar_dex/robot/assets/allegro_hand_right_no_collision.urdf"
# sapien setup
engine = sapien.Engine()
scene_config = sapien.SceneConfig()
scene_config.gravity = [0, 0, 0] # type: ignore
scene_config.disable_collision_visual = True
scene_config.enable_adaptive_force = False
scene = engine.create_scene(scene_config)

loader = scene.create_urdf_loader()
robot: sapien.Articulation = loader.load(hand_urdf_path)

[2024-01-18 16:27:07.690] [SAPIEN] [warning] A second engine will share the same internal structures with the first one. Arguments passed to constructor will be ignored.


In [36]:
# get joint names
joints = robot.get_active_joints()
[joint.name for joint in joints]

['joint_0.0',
 'joint_1.0',
 'joint_2.0',
 'joint_3.0',
 'joint_4.0',
 'joint_5.0',
 'joint_6.0',
 'joint_7.0',
 'joint_8.0',
 'joint_9.0',
 'joint_10.0',
 'joint_11.0',
 'joint_12.0',
 'joint_13.0',
 'joint_14.0',
 'joint_15.0']

In [37]:
robot.get_qlimits()

array([[-0.47  ,  0.47  ],
       [-0.196 ,  1.61  ],
       [-0.174 ,  1.709 ],
       [-0.227 ,  1.618 ],
       [-0.47  ,  0.47  ],
       [-0.196 ,  1.61  ],
       [-0.174 ,  1.709 ],
       [-0.227 ,  1.618 ],
       [-0.47  ,  0.47  ],
       [-0.196 ,  1.61  ],
       [-0.174 ,  1.709 ],
       [-0.227 ,  1.618 ],
       [ 0.263 ,  1.396 ],
       [-0.1   ,  1.16  ],
       [-0.189 ,  1.222 ],
       [-0.162 ,  0.7854]], dtype=float32)

In [38]:
[link.get_name() for link in robot.get_links()]

['base_link',
 'link_0.0',
 'link_1.0',
 'link_2.0',
 'link_3.0',
 'link_3.0_tip',
 'link_4.0',
 'link_5.0',
 'link_6.0',
 'link_7.0',
 'link_7.0_tip',
 'link_8.0',
 'link_9.0',
 'link_10.0',
 'link_11.0',
 'link_11.0_tip',
 'link_12.0',
 'link_13.0',
 'link_14.0',
 'link_15.0',
 'link_15.0_tip',
 'palm']

In [58]:
import torch

qpos=np.zeros(4)
x = np.array([1., 2., 3.])
ctr_qpos = x.copy()
torch_x = torch.as_tensor(ctr_qpos)
torch_x.requires_grad_(True)
torch_qpos = torch.as_tensor(qpos)
torch_qpos[:3] = torch_x

loss = torch.sum(torch_qpos)
loss.backward()
print(torch_x.grad)

tensor([1., 1., 1.], dtype=torch.float64)
